# V_CRG_STUDENT_COURSE — Preprocessing V5

Scope:
- Work only on `V_CRG_STUDENT_COURSE`.
- Load raw Parquet.
- Preserve raw data and original ID values.
- `is_last_try` is not used because it was removed from the source query.
- Calculate attempts manually per student-course.
- Do not overwrite original IDs.
- Create `*_raw`, `*_base`, `*_suffix`, `*_key` for ID columns.
- Do not assume pass/fail from `final_mark`.
- Official outcome comes only from `finish_status`.
- Apply explicit null-handling rules before creating the clean dataset.
- Do not build models.
- Do not build course difficulty.
- Do not build recommendation logic.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import (
    RAW_DIR,
    CLEAN_DIR,
    FEATURES_DIR as BASE_FEATURES_DIR,
    REPORTS_DIR as BASE_REPORTS_DIR,
    ensure_dir,
)
from src.cleaning_utils import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

VERSION = "v5"

RAW_PATH = RAW_DIR / "v_crg_student_course_raw.parquet"
PREPROCESSED_DIR = ensure_dir(CLEAN_DIR / "V_CRG_STUDENT_COURSE")
FEATURES_DIR = ensure_dir(BASE_FEATURES_DIR / "V_CRG_STUDENT_COURSE")
REPORTS_DIR = ensure_dir(BASE_REPORTS_DIR / "V_CRG_STUDENT_COURSE")

print("VERSION:", VERSION)
print("RAW_PATH:", RAW_PATH)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("FEATURES_DIR:", FEATURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

assert RAW_PATH.exists(), f"Raw file not found: {RAW_PATH}"


## 1. Load raw Parquet file

This step loads the raw V5 Parquet file.  
No source values are changed here.


In [ ]:
df_raw_loaded_v5 = pd.read_parquet(RAW_PATH)

print("Raw loaded shape:", df_raw_loaded_v5.shape)
display(df_raw_loaded_v5.head())
display(df_raw_loaded_v5.dtypes)

In [ ]:
assert isinstance(df_raw_loaded_v5, pd.DataFrame)
assert len(df_raw_loaded_v5) > 0, "Raw dataframe is empty."

print("Validation passed: raw Parquet loaded successfully.")

## 2. Standardize column names and validate required columns

Only column names are standardized to lowercase.  
Raw values remain unchanged.

Important:
- `is_last_try` is not required and is not used.


In [ ]:
df_raw_v5 = df_raw_loaded_v5.copy() ##  createa copy df

df_raw_v5.columns = (
    df_raw_v5.columns
    .astype(str)
    .str.strip()
    .str.lower()
)

required_raw_columns_v5 = [
    "student_course_id",
    "student_id",
    "course_id",
    "part_id",
    "grade_id",
    "final_mark",
    "points",
    "finish_status",
    "course_name_sl",
    "study_mode",
    "degree_id",
    "degree_name_sl",
    "faculty_id",
    "course_credits",
    "active",
]

missing_required_columns_v5 = sorted(set(required_raw_columns_v5) - set(df_raw_v5.columns))
extra_columns_v5 = sorted(set(df_raw_v5.columns) - set(required_raw_columns_v5))

print("Missing required columns:", missing_required_columns_v5)
print("Extra source columns:", extra_columns_v5)

assert len(missing_required_columns_v5) == 0, f"Missing columns: {missing_required_columns_v5}"

df_raw_v5 = df_raw_v5[required_raw_columns_v5].copy() ## re assign with the required column 

print("Raw selected shape:", df_raw_v5.shape)
display(df_raw_v5.head())

In [ ]:
assert list(df_raw_v5.columns) == required_raw_columns_v5
assert df_raw_v5.shape[1] == len(required_raw_columns_v5)
assert "is_last_try" not in df_raw_v5.columns, "is_last_try should not be used in V5 workflow."

print("Validation passed: all required columns exist and selected.")

## 3. Save raw profile report

This report gives a quick overview of row count, dtype, nulls, and unique values per column.


In [ ]:
raw_profile_report_v5 = pd.DataFrame({ ## create a profile report for the raw data from the copied df
    "column": df_raw_v5.columns,
    "dtype": [str(df_raw_v5[col].dtype) for col in df_raw_v5.columns],
    "row_count": len(df_raw_v5),
    "non_null_count": [int(df_raw_v5[col].notna().sum()) for col in df_raw_v5.columns],
    "null_count": [int(df_raw_v5[col].isna().sum()) for col in df_raw_v5.columns],
    "null_ratio": [float(df_raw_v5[col].isna().mean()) for col in df_raw_v5.columns],
    "unique_count": [int(df_raw_v5[col].nunique(dropna=True)) for col in df_raw_v5.columns],
})

raw_profile_report_path_v5 = REPORTS_DIR / "raw_profile_report_v5.csv"
raw_profile_report_v5.to_csv(raw_profile_report_path_v5, index=False, encoding="utf-8-sig")

display(raw_profile_report_v5)
print("Saved:", raw_profile_report_path_v5)

In [ ]:
assert raw_profile_report_path_v5.exists()
assert len(raw_profile_report_v5) == len(required_raw_columns_v5)

print("Validation passed: raw profile report saved.")

## 4. Save null report

This report shows missing values per column.  
It is important before deciding archive/drop or critical issues.


In [ ]:
null_report_v5 = (
    df_raw_v5  ## Calculate null counts by column and the output is series while the coulumn become the index so ireset the index and give it the name needed
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "null_count"})
)

null_report_v5["row_count"] = len(df_raw_v5)
null_report_v5["null_ratio"] = null_report_v5["null_count"] / len(df_raw_v5)

null_report_v5 = null_report_v5.sort_values("null_count", ascending=False).reset_index(drop=True)

null_report_path_v5 = REPORTS_DIR / "null_report_v5.csv"
null_report_v5.to_csv(null_report_path_v5, index=False, encoding="utf-8-sig")

display(null_report_v5)
print("Saved:", null_report_path_v5)

In [ ]:
assert null_report_path_v5.exists()
assert set(null_report_v5["column"]) == set(required_raw_columns_v5)

print("Validation passed: null report saved.")

In [ ]:
plot_df = null_report_v5.sort_values("null_ratio", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["column"], plot_df["null_ratio"])
plt.title("Null Ratio by Column — V_CRG_STUDENT_COURSE V5")
plt.xlabel("Null Ratio")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

## 5. Save finish_status distribution report

`finish_status` is the only official source for academic outcome.  
`final_mark >= 50` is not used as pass logic.


In [ ]:
finish_status_distribution_v5 = (
    df_raw_v5["finish_status"] ## take only one column and make it a string type and strip ,upper ut ...
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("<NULL>")
    .value_counts(dropna=False)
    .reset_index()
)

finish_status_distribution_v5.columns = ["finish_status", "count"]
finish_status_distribution_v5["row_count"] = len(df_raw_v5)
finish_status_distribution_v5["ratio"] = finish_status_distribution_v5["count"] / len(df_raw_v5)

finish_status_distribution_path_v5 = REPORTS_DIR / "finish_status_distribution_v5.csv"
finish_status_distribution_v5.to_csv(
    finish_status_distribution_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(finish_status_distribution_v5)
print("Saved:", finish_status_distribution_path_v5)

In [ ]:
assert finish_status_distribution_path_v5.exists()
assert finish_status_distribution_v5["count"].sum() == len(df_raw_v5)

print("Validation passed: finish_status distribution saved.")

In [ ]:
plot_df = finish_status_distribution_v5.sort_values("count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["finish_status"], plot_df["count"])
plt.title("finish_status Distribution — V5")
plt.xlabel("Rows")
plt.ylabel("finish_status")
plt.tight_layout()
plt.show()

## 6. Numeric validation without stopping the notebook

Rules:
- `final_mark` should be integer-like.
- `course_credits` may contain unexpected fractional values; do not round automatically.
- `points` is float and remains raw.

Important:
- If `course_credits` has fractional values, report them and continue.
- Do not manually fill or modify `course_credits`.


In [ ]:
numeric_validation_report_v5 = pd.DataFrame([
    integer_like_report(df_raw_v5, "final_mark"),
    integer_like_report(df_raw_v5, "course_credits"),
])

points_numeric_v5 = pd.to_numeric(df_raw_v5["points"], errors="coerce")

points_report_v5 = pd.DataFrame([{
    "column": "points",
    "source_dtype": str(df_raw_v5["points"].dtype),
    "non_null_count": int(df_raw_v5["points"].notna().sum()),
    "numeric_count": int(points_numeric_v5.notna().sum()),
    "non_numeric_or_null_count": int(df_raw_v5["points"].shape[0] - points_numeric_v5.notna().sum()),
    "fractional_count": np.nan,
    "fractional_ratio": np.nan,
}])

numeric_validation_report_v5 = pd.concat(
    [numeric_validation_report_v5, points_report_v5],
    ignore_index=True
)

numeric_validation_report_path_v5 = REPORTS_DIR / "numeric_validation_report_v5.csv"
numeric_validation_report_v5.to_csv(
    numeric_validation_report_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(numeric_validation_report_v5)
print("Saved:", numeric_validation_report_path_v5)


In [ ]:
final_mark_fractional = numeric_validation_report_v5.loc[
    numeric_validation_report_v5["column"].eq("final_mark"),
    "fractional_count"
].iloc[0]

course_credits_fractional = numeric_validation_report_v5.loc[
    numeric_validation_report_v5["column"].eq("course_credits"),
    "fractional_count"
].iloc[0]

assert final_mark_fractional == 0, "final_mark has fractional values."

if course_credits_fractional > 0:
    print("WARNING: course_credits has fractional values.")
    print("These rows will be reported and flagged. No rounding will be applied.")
else:
    print("course_credits is integer-like.")

assert numeric_validation_report_path_v5.exists()

print("Validation passed: final_mark is integer-like. course_credits inspected.")

## 7. Inspect fractional course_credits

These rows are saved for inspection instead of being silently changed.


In [ ]:
course_credits_numeric_v5 = pd.to_numeric(df_raw_v5["course_credits"], errors="coerce")

fractional_course_credits_mask_v5 = (
    course_credits_numeric_v5.notna()
    & ((course_credits_numeric_v5 % 1) != 0)
)

fractional_course_credits_rows_v5 = df_raw_v5[fractional_course_credits_mask_v5].copy()

fractional_course_credits_distribution_v5 = (
    fractional_course_credits_rows_v5["course_credits"]
    .value_counts(dropna=False)
    .reset_index()
)

fractional_course_credits_distribution_v5.columns = ["course_credits", "count"]
fractional_course_credits_distribution_v5["row_count"] = len(df_raw_v5)
fractional_course_credits_distribution_v5["ratio"] = (
    fractional_course_credits_distribution_v5["count"] / len(df_raw_v5)
)

fractional_course_credits_rows_path_v5 = REPORTS_DIR / "fractional_course_credits_rows_v5.csv"
fractional_course_credits_distribution_path_v5 = (
    REPORTS_DIR / "fractional_course_credits_distribution_v5.csv"
)

fractional_course_credits_rows_v5.to_csv(
    fractional_course_credits_rows_path_v5,
    index=False,
    encoding="utf-8-sig"
)

fractional_course_credits_distribution_v5.to_csv(
    fractional_course_credits_distribution_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(fractional_course_credits_distribution_v5)
display(fractional_course_credits_rows_v5.head(50))

print("Fractional course_credits rows:", len(fractional_course_credits_rows_v5))
print("Saved:", fractional_course_credits_rows_path_v5)
print("Saved:", fractional_course_credits_distribution_path_v5)

In [ ]:
assert fractional_course_credits_rows_path_v5.exists()
assert fractional_course_credits_distribution_path_v5.exists()

print("Validation passed: fractional course_credits rows reported.")

## 8. Drop exact duplicate rows from working dataframe

Raw data remains unchanged in `df_raw_v5`.  
Exact duplicate rows are reported, then removed only from `df_work_v5`.


In [ ]:
duplicate_full_row_mask_v5 = df_raw_v5.duplicated(keep="first")

df_duplicate_rows_v5 = df_raw_v5[duplicate_full_row_mask_v5].copy()

duplicate_rows_path_v5 = REPORTS_DIR / "duplicate_rows_v5.csv"
df_duplicate_rows_v5.to_csv(duplicate_rows_path_v5, index=False, encoding="utf-8-sig")

df_work_v5 = df_raw_v5.drop_duplicates(keep="first").copy()  #### define the work df *** *** *** freedom 
df_work_v5["_source_row_number_v5"] = np.arange(len(df_work_v5)) ### like a new index

print("Raw rows before duplicate removal:", len(df_raw_v5))
print("Exact duplicate rows removed:", len(df_duplicate_rows_v5))
print("Rows after duplicate removal:", len(df_work_v5))
print("Saved:", duplicate_rows_path_v5)

display(df_duplicate_rows_v5.head(50))

In [ ]:
assert len(df_work_v5) + len(df_duplicate_rows_v5) == len(df_raw_v5)
assert df_work_v5[required_raw_columns_v5].duplicated().sum() == 0
assert duplicate_rows_path_v5.exists()

print("Validation passed: exact duplicates reported and removed from working dataframe.")

## 9. Define finish_status and null-handling rules

Current decision:

Keep:
- `P`
- `F`
- `FA`
- `FE`
- `W`

Archive/drop:
- `T`
- `X`
- `IP`
- `L`
- `Z`
- `D`
- `I`
- `ST`
- null `finish_status`

Also:
- Treat `F`, `FA`, and `FE` as the same fail family.
- `final_mark` null can be filled with 0 only for `F`, `FA`, `FE`, `W`, `IP`, `Z`.
- `points` remains raw; only its range is inspected.
- `grade_id` remains raw; do not interpret it without `grade_version_id`.
- `course_credits` null is not manually filled.
- `course_credits == 24` is archive/drop.
- `course_credits == 0` is archive/drop in this notebook by default.


In [ ]:
KEEP_STATUSES_V5 = {"P", "F", "FA", "FE", "W"}

PASS_STATUSES_V5 = {"P"}

FAIL_STATUSES_V5 = {"F", "FA", "FE"}

WITHDRAWN_STATUSES_V5 = {"W"}

DROP_ARCHIVE_STATUSES_V5 = {"T", "X", "IP", "L", "Z", "D", "I", "ST"}

FINAL_MARK_ZERO_FILL_STATUSES_V5 = {"F", "FA", "FE", "W", "IP", "Z"}

KNOWN_STATUSES_V5 = KEEP_STATUSES_V5 | DROP_ARCHIVE_STATUSES_V5

DROP_ZERO_COURSE_CREDITS_V5 = True

print("Keep statuses:", sorted(KEEP_STATUSES_V5))
print("Pass statuses:", sorted(PASS_STATUSES_V5))
print("Fail statuses:", sorted(FAIL_STATUSES_V5))
print("Withdrawn statuses:", sorted(WITHDRAWN_STATUSES_V5))
print("Drop/archive statuses:", sorted(DROP_ARCHIVE_STATUSES_V5))
print("Final mark zero-fill statuses:", sorted(FINAL_MARK_ZERO_FILL_STATUSES_V5))
print("DROP_ZERO_COURSE_CREDITS_V5:", DROP_ZERO_COURSE_CREDITS_V5)

In [ ]:
assert KEEP_STATUSES_V5 == {"P", "F", "FA", "FE", "W"}
assert FAIL_STATUSES_V5 == {"F", "FA", "FE"}
assert "T" in DROP_ARCHIVE_STATUSES_V5
assert "IP" in DROP_ARCHIVE_STATUSES_V5
assert "W" in KEEP_STATUSES_V5
assert "Z" in FINAL_MARK_ZERO_FILL_STATUSES_V5

print("Validation passed: finish_status and null-handling rules are active.")

## 10. Create flagged dataframe after duplicate removal

This dataframe is the base for null handling, cleaning, and reporting.  
Original source columns are preserved.


In [ ]:
df_flagged_v5 = df_work_v5.copy() ## flag df 

df_flagged_v5["finish_status_clean"] = (
    df_flagged_v5["finish_status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_flagged_v5["final_mark_numeric_v5"] = (
    pd.to_numeric(df_flagged_v5["final_mark"], errors="coerce")
    .astype("float64")
)

df_flagged_v5["points_float"] = (
    pd.to_numeric(df_flagged_v5["points"], errors="coerce")
    .astype("float64")
)

df_flagged_v5["course_credits_num"] = (
    pd.to_numeric(df_flagged_v5["course_credits"], errors="coerce")
    .astype("float64")
)
 
df_flagged_v5["course_credits_int"] = np.where(
    df_flagged_v5["course_credits_num"].notna()
    & ((df_flagged_v5["course_credits_num"] % 1) == 0),
    df_flagged_v5["course_credits_num"],
    np.nan
)

df_flagged_v5["course_credits_int"] = (
    pd.Series(df_flagged_v5["course_credits_int"], index=df_flagged_v5.index)
    .astype("Int64")
)

df_flagged_v5["has_fractional_course_credits"] = (
    df_flagged_v5["course_credits_num"].notna()
    & ((df_flagged_v5["course_credits_num"] % 1) != 0)
)

df_flagged_v5["course_name_clean"] = (
    df_flagged_v5["course_name_sl"]
    .astype("string")
    .str.strip()
)

df_flagged_v5["degree_name_clean"] = (
    df_flagged_v5["degree_name_sl"]
    .astype("string")
    .str.strip()
)

df_flagged_v5["study_mode_clean"] = (
    df_flagged_v5["study_mode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

display(df_flagged_v5.head(20))

In [ ]:
assert len(df_flagged_v5) == len(df_work_v5)
assert "finish_status_clean" in df_flagged_v5.columns
assert "course_credits_num" in df_flagged_v5.columns
assert "has_fractional_course_credits" in df_flagged_v5.columns
assert "is_last_try" not in df_flagged_v5.columns

print("Validation passed: flagged dataframe created after duplicate removal.")

## 11. Apply explicit null-handling flags

Rules:
1. Any row with null `finish_status` is archive/drop.
2. If `finish_status` is null even with `final_mark = 0`, it is still archive/drop.
3. `final_mark` null is filled with 0 only for statuses `F`, `FA`, `FE`, `W`, `IP`, `Z`.
4. `points` remains raw; only range is inspected.
5. `grade_id` remains raw and is not interpreted.
6. `course_credits` null is not filled manually.
7. `course_name_sl` and `degree_name_sl` do not cause deletion directly.


In [ ]:
df_flagged_v5["finish_status_is_null_v5"] = df_flagged_v5["finish_status_clean"].isna() ## create flags 

df_flagged_v5["finish_status_null_without_grade_mark_points_v5"] = (
    df_flagged_v5["finish_status_is_null_v5"]
    & df_flagged_v5["grade_id"].isna()
    & df_flagged_v5["final_mark"].isna()
    & df_flagged_v5["points"].isna()
)

df_flagged_v5["finish_status_null_with_final_mark_zero_v5"] = (
    df_flagged_v5["finish_status_is_null_v5"]
    & df_flagged_v5["final_mark_numeric_v5"].eq(0)
)

df_flagged_v5["final_mark_was_filled_zero_v5"] = (
    df_flagged_v5["final_mark_numeric_v5"].isna()
    & df_flagged_v5["finish_status_clean"].isin(FINAL_MARK_ZERO_FILL_STATUSES_V5)
)

df_flagged_v5["final_mark_clean_v5"] = df_flagged_v5["final_mark_numeric_v5"]

df_flagged_v5.loc[
    df_flagged_v5["final_mark_was_filled_zero_v5"],
    "final_mark_clean_v5"
] = 0

df_flagged_v5["final_mark_clean_int_v5"] = (
    df_flagged_v5["final_mark_clean_v5"]
    .round()
    .astype("Int64")
)

df_flagged_v5["course_credits_is_null_v5"] = df_flagged_v5["course_credits_num"].isna()
df_flagged_v5["course_name_is_null_v5"] = df_flagged_v5["course_name_sl"].isna()
df_flagged_v5["degree_name_is_null_v5"] = df_flagged_v5["degree_name_sl"].isna()

display(df_flagged_v5[[
    "finish_status",
    "finish_status_clean",
    "grade_id",
    "final_mark",
    "final_mark_numeric_v5",
    "final_mark_clean_v5",
    "final_mark_was_filled_zero_v5",
    "points",
    "points_float",
    "course_credits",
    "course_credits_num",
    "course_credits_is_null_v5",
]].head(30))

In [ ]:
invalid_final_mark_fill_v5 = df_flagged_v5[
    df_flagged_v5["final_mark_was_filled_zero_v5"]
    & ~df_flagged_v5["finish_status_clean"].isin(FINAL_MARK_ZERO_FILL_STATUSES_V5)
]

assert len(invalid_final_mark_fill_v5) == 0, "final_mark was filled outside allowed statuses."

null_handling_report_v5 = pd.DataFrame([
    {
        "case": "finish_status_null_total",
        "count": int(df_flagged_v5["finish_status_is_null_v5"].sum()),
    },
    {
        "case": "finish_status_null_without_grade_mark_points",
        "count": int(df_flagged_v5["finish_status_null_without_grade_mark_points_v5"].sum()),
    },
    {
        "case": "finish_status_null_with_final_mark_zero",
        "count": int(df_flagged_v5["finish_status_null_with_final_mark_zero_v5"].sum()),
    },
    {
        "case": "final_mark_null_filled_zero_allowed_statuses",
        "count": int(df_flagged_v5["final_mark_was_filled_zero_v5"].sum()),
    },
    {
        "case": "course_credits_null_not_filled",
        "count": int(df_flagged_v5["course_credits_is_null_v5"].sum()),
    },
    {
        "case": "course_name_null_not_direct_drop",
        "count": int(df_flagged_v5["course_name_is_null_v5"].sum()),
    },
    {
        "case": "degree_name_null_not_direct_drop",
        "count": int(df_flagged_v5["degree_name_is_null_v5"].sum()),
    },
])

null_handling_report_v5["row_count"] = len(df_flagged_v5)
null_handling_report_v5["ratio"] = null_handling_report_v5["count"] / len(df_flagged_v5)

null_handling_report_path_v5 = REPORTS_DIR / "null_handling_report_v5.csv"
null_handling_report_v5.to_csv(null_handling_report_path_v5, index=False, encoding="utf-8-sig")

display(null_handling_report_v5)
print("Saved:", null_handling_report_path_v5)

assert null_handling_report_path_v5.exists()

print("Validation passed: null-handling rules applied and reported.")

## 12. Inspect points range

`points` remains raw.  
This step only reports its numeric range and suspicious negative values.


In [ ]:
points_range_report_v5 = pd.DataFrame([{
    "metric": "points_float",
    "count": int(df_flagged_v5["points_float"].notna().sum()),
    "null_count": int(df_flagged_v5["points_float"].isna().sum()),
    "min": float(df_flagged_v5["points_float"].min()) if df_flagged_v5["points_float"].notna().any() else np.nan,
    "max": float(df_flagged_v5["points_float"].max()) if df_flagged_v5["points_float"].notna().any() else np.nan,
    "mean": float(df_flagged_v5["points_float"].mean()) if df_flagged_v5["points_float"].notna().any() else np.nan,
    "negative_count": int(df_flagged_v5["points_float"].lt(0).sum()),
}])

points_range_report_path_v5 = REPORTS_DIR / "points_range_report_v5.csv"
points_range_report_v5.to_csv(points_range_report_path_v5, index=False, encoding="utf-8-sig")

negative_points_rows_v5 = df_flagged_v5[df_flagged_v5["points_float"].lt(0)].copy() ## larger than 
negative_points_rows_path_v5 = REPORTS_DIR / "negative_points_rows_v5.csv"
negative_points_rows_v5.to_csv(negative_points_rows_path_v5, index=False, encoding="utf-8-sig")

display(points_range_report_v5)
print("Saved:", points_range_report_path_v5)
print("Saved:", negative_points_rows_path_v5)

In [ ]:
assert points_range_report_path_v5.exists()
assert negative_points_rows_path_v5.exists()

print("Validation passed: points range inspected without modifying raw points.")

## 13. Create academic flags from finish_status

Important:
- `P` = pass.
- `F`, `FA`, `FE` = fail family.
- `W` = withdrawn, kept in clean data but not internal performance.
- `T`, `X`, `IP`, `L`, `Z`, `D`, `I`, `ST` = archive/drop.


In [ ]:
df_flagged_v5["is_official_pass"] = (
    df_flagged_v5["finish_status_clean"].isin(PASS_STATUSES_V5)
)

df_flagged_v5["is_transfer_pass"] = False

df_flagged_v5["is_internal_performance_attempt"] = (
    df_flagged_v5["finish_status_clean"].isin(PASS_STATUSES_V5 | FAIL_STATUSES_V5)
)

df_flagged_v5["is_fail"] = (
    df_flagged_v5["finish_status_clean"].isin(FAIL_STATUSES_V5)
)

df_flagged_v5["is_fail_like"] = (
    df_flagged_v5["finish_status_clean"].isin({"FA", "FE"})
)

df_flagged_v5["is_withdrawn"] = (
    df_flagged_v5["finish_status_clean"].isin(WITHDRAWN_STATUSES_V5)
)

df_flagged_v5["is_in_progress"] = False

df_flagged_v5["is_unknown_finish_status"] = (
    df_flagged_v5["finish_status_clean"].notna()
    & ~df_flagged_v5["finish_status_clean"].isin(KNOWN_STATUSES_V5)
)

display(df_flagged_v5[[
    "finish_status",
    "finish_status_clean",
    "is_official_pass",
    "is_transfer_pass",
    "is_internal_performance_attempt",
    "is_fail",
    "is_fail_like",
    "is_withdrawn",
    "is_in_progress",
    "is_unknown_finish_status",
]].head(30))

In [ ]:
assert not df_flagged_v5["is_transfer_pass"].any()
assert not (df_flagged_v5["is_official_pass"] & df_flagged_v5["is_fail"]).any()
assert not (df_flagged_v5["is_withdrawn"] & df_flagged_v5["is_internal_performance_attempt"]).any()

print("Validation passed: academic flags are consistent.")

## 14. Flag archive/drop rows

Rows are not silently removed.  
They are flagged and saved in reports first.


In [ ]:
df_flagged_v5["archive_drop_reason_v5"] = "" ### create a new column for the reason of drop or archive 


append_reason(
    df_flagged_v5,
    df_flagged_v5["finish_status_null_without_grade_mark_points_v5"],
    "archive_drop_reason_v5",
    "finish_status_null_without_grade_mark_points_archive_drop",
)

append_reason(
    df_flagged_v5,
    df_flagged_v5["finish_status_null_with_final_mark_zero_v5"],
    "archive_drop_reason_v5",
    "finish_status_null_even_with_final_mark_zero_archive_drop",
)

append_reason(
    df_flagged_v5,
    df_flagged_v5["finish_status_is_null_v5"],
    "archive_drop_reason_v5",
    "finish_status_null_archive_drop",
)

append_reason(
    df_flagged_v5,
    df_flagged_v5["finish_status_clean"].isin(DROP_ARCHIVE_STATUSES_V5),
    "archive_drop_reason_v5",
    "finish_status_archive_drop_by_business_rule",
)

append_reason(
    df_flagged_v5,
    df_flagged_v5["course_credits_num"].eq(24),
    "archive_drop_reason_v5",
    "course_credits_24_archive_drop",
)

if DROP_ZERO_COURSE_CREDITS_V5:
    append_reason(
        df_flagged_v5,
        df_flagged_v5["course_credits_num"].eq(0),
        "archive_drop_reason_v5",
        "course_credits_0_archive_drop",
    )

append_reason(
    df_flagged_v5,
    df_flagged_v5["is_unknown_finish_status"],
    "archive_drop_reason_v5",
    "unknown_finish_status_review_archive_drop",
)

df_flagged_v5["is_archive_or_drop"] = (
    df_flagged_v5["archive_drop_reason_v5"]
    .astype("string")
    .str.len()
    .gt(0)
)

archive_drop_rows_v5 = df_flagged_v5[df_flagged_v5["is_archive_or_drop"]].copy()

archive_drop_report_v5 = (
    archive_drop_rows_v5["archive_drop_reason_v5"]
    .value_counts(dropna=False)
    .reset_index()
)

archive_drop_report_v5.columns = ["archive_drop_reason_v5", "count"]
archive_drop_report_v5["row_count"] = len(df_flagged_v5)
archive_drop_report_v5["ratio"] = archive_drop_report_v5["count"] / len(df_flagged_v5)

archive_drop_report_path_v5 = REPORTS_DIR / "archive_drop_report_v5.csv"
archive_drop_rows_path_v5 = REPORTS_DIR / "archive_drop_rows_v5.csv"

archive_drop_report_v5.to_csv(
    archive_drop_report_path_v5,
    index=False,
    encoding="utf-8-sig"
)

archive_drop_rows_v5.to_csv(
    archive_drop_rows_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(archive_drop_report_v5)
print("Archive/drop rows:", len(archive_drop_rows_v5))
print("Saved:", archive_drop_report_path_v5)
print("Saved:", archive_drop_rows_path_v5)


In [ ]:
assert archive_drop_report_path_v5.exists()
assert archive_drop_rows_path_v5.exists()
assert len(archive_drop_rows_v5) == int(df_flagged_v5["is_archive_or_drop"].sum())

print("Validation passed: archive/drop rows flagged and reported.")

In [ ]:
if len(archive_drop_report_v5) > 0:
    plot_df = archive_drop_report_v5.sort_values("count", ascending=True)

    plt.figure(figsize=(12, 8))
    plt.barh(plot_df["archive_drop_reason_v5"], plot_df["count"])
    plt.title("Archive / Drop Reasons — V5")
    plt.xlabel("Rows")
    plt.ylabel("Reason")
    plt.tight_layout()
    plt.show()
else:
    print("No archive/drop rows to visualize.")

## 15. Flag critical issues

Critical issues:
- missing `student_id`
- missing `course_id`
- missing `part_id`

Without these IDs, we cannot safely create student-course attempts or current status.


In [ ]:
df_flagged_v5["critical_issue_reason_v5"] = ""

append_reason(
    df_flagged_v5,
    df_flagged_v5["student_id"].isna(),
    "critical_issue_reason_v5",
    "missing_student_id_critical",
)

append_reason(
    df_flagged_v5,
    df_flagged_v5["course_id"].isna(),
    "critical_issue_reason_v5",
    "missing_course_id_critical",
)

append_reason(
    df_flagged_v5,
    df_flagged_v5["part_id"].isna(),
    "critical_issue_reason_v5",
    "missing_part_id_critical",
)

df_flagged_v5["is_critical_issue"] = (
    df_flagged_v5["critical_issue_reason_v5"]
    .astype("string")
    .str.len()
    .gt(0)
)

critical_issue_rows_v5 = df_flagged_v5[df_flagged_v5["is_critical_issue"]].copy()

critical_issues_report_v5 = (
    critical_issue_rows_v5["critical_issue_reason_v5"]
    .value_counts(dropna=False)
    .reset_index()
)

critical_issues_report_v5.columns = ["critical_issue_reason_v5", "count"]
critical_issues_report_v5["row_count"] = len(df_flagged_v5)
critical_issues_report_v5["ratio"] = critical_issues_report_v5["count"] / len(df_flagged_v5)

critical_issues_report_path_v5 = REPORTS_DIR / "critical_issues_report_v5.csv"
critical_issue_rows_path_v5 = REPORTS_DIR / "critical_issue_rows_v5.csv"

critical_issues_report_v5.to_csv(
    critical_issues_report_path_v5,
    index=False,
    encoding="utf-8-sig"
)

critical_issue_rows_v5.to_csv(
    critical_issue_rows_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(critical_issues_report_v5)
print("Critical rows:", len(critical_issue_rows_v5))
print("Saved:", critical_issues_report_path_v5)
print("Saved:", critical_issue_rows_path_v5)

In [ ]:
assert critical_issues_report_path_v5.exists()
assert critical_issue_rows_path_v5.exists()
assert len(critical_issue_rows_v5) == int(df_flagged_v5["is_critical_issue"].sum())

print("Validation passed: critical issue rows flagged and reported.")

## 16. Report course_credits null rows without filling them

`course_credits` null values are not filled manually.  
They are reported for data-quality follow-up.


In [ ]:
course_credits_null_rows_v5 = df_flagged_v5[df_flagged_v5["course_credits_is_null_v5"]].copy()

course_credits_null_rows_path_v5 = REPORTS_DIR / "course_credits_null_rows_v5.csv"
course_credits_null_rows_v5.to_csv(
    course_credits_null_rows_path_v5,
    index=False,
    encoding="utf-8-sig"
)

course_credits_null_report_v5 = pd.DataFrame([{
    "case": "course_credits_null_not_filled",
    "count": len(course_credits_null_rows_v5),
    "row_count": len(df_flagged_v5),
    "ratio": len(course_credits_null_rows_v5) / len(df_flagged_v5),
}])

course_credits_null_report_path_v5 = REPORTS_DIR / "course_credits_null_report_v5.csv"
course_credits_null_report_v5.to_csv(
    course_credits_null_report_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(course_credits_null_report_v5)
display(course_credits_null_rows_v5.head(50))

print("Saved:", course_credits_null_rows_path_v5)
print("Saved:", course_credits_null_report_path_v5)

In [ ]:
assert course_credits_null_rows_path_v5.exists()
assert course_credits_null_report_path_v5.exists()

print("Validation passed: course_credits null rows reported without manual filling.")

## 17. Create df_attempts_clean_v5

Clean attempts keep only:
- `P`
- `F`
- `FA`
- `FE`
- `W`

And remove/isolate:
- exact duplicates
- archive/drop statuses
- `course_credits == 24`
- `course_credits == 0`
- null `finish_status`
- critical missing IDs


In [ ]:
clean_keep_mask_v5 = (
    ~df_flagged_v5["is_archive_or_drop"]
    & ~df_flagged_v5["is_critical_issue"]
    & df_flagged_v5["finish_status_clean"].isin(KEEP_STATUSES_V5)
)

df_attempts_clean_v5 = df_flagged_v5[clean_keep_mask_v5].copy()

removed_or_isolated_count_v5 = len(df_flagged_v5) - len(df_attempts_clean_v5)

print("Raw rows:", len(df_raw_v5))
print("Exact duplicate rows removed:", len(df_duplicate_rows_v5))
print("Rows after duplicates:", len(df_work_v5))
print("Clean rows:", len(df_attempts_clean_v5))
print("Removed / isolated after duplicate step:", removed_or_isolated_count_v5)

display(
    df_attempts_clean_v5["finish_status_clean"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "finish_status_clean", "finish_status_clean": "count"})
)

In [ ]:
assert df_attempts_clean_v5["finish_status_clean"].isin(KEEP_STATUSES_V5).all()
assert not df_attempts_clean_v5["is_archive_or_drop"].any()
assert not df_attempts_clean_v5["is_critical_issue"].any()
assert not df_attempts_clean_v5["course_credits_num"].eq(24).any()

if DROP_ZERO_COURSE_CREDITS_V5:
    assert not df_attempts_clean_v5["course_credits_num"].eq(0).any()

assert df_attempts_clean_v5["student_id"].notna().all()
assert df_attempts_clean_v5["course_id"].notna().all()
assert df_attempts_clean_v5["part_id"].notna().all()

print("Validation passed: df_attempts_clean_v5 created using updated rules.")

## 18. Define ID normalization helpers

For every ID column, we will create:
- `*_raw`
- `*_base`
- `*_suffix`
- `*_key`

Original ID columns are not overwritten.


In [ ]:
ID_COLUMNS_V5 = [
    "student_course_id",
    "student_id",
    "course_id",
    "part_id",
    "grade_id",
    "degree_id",
    "faculty_id",
]


print("ID columns:", ID_COLUMNS_V5)


In [ ]:
for col in ID_COLUMNS_V5:
    assert col in df_attempts_clean_v5.columns, f"Missing ID column: {col}"

print("Validation passed: ID normalization helpers ready.")

## 19. Create df_attempts_normalized_v5

This dataframe starts from `df_attempts_clean_v5` and adds normalized ID columns.  
Original IDs remain preserved.


In [ ]:
df_attempts_normalized_v5 = df_attempts_clean_v5.copy()

for id_col in ID_COLUMNS_V5:
    df_attempts_normalized_v5 = add_id_components(df_attempts_normalized_v5, id_col)

id_display_columns_v5 = (
    ID_COLUMNS_V5
    + [f"{col}_raw" for col in ID_COLUMNS_V5]
    + [f"{col}_base" for col in ID_COLUMNS_V5]
    + [f"{col}_suffix" for col in ID_COLUMNS_V5]
    + [f"{col}_key" for col in ID_COLUMNS_V5]
)

display(df_attempts_normalized_v5[id_display_columns_v5].head())

In [ ]:
for id_col in ID_COLUMNS_V5:
    assert id_col in df_attempts_normalized_v5.columns
    assert f"{id_col}_raw" in df_attempts_normalized_v5.columns
    assert f"{id_col}_base" in df_attempts_normalized_v5.columns
    assert f"{id_col}_suffix" in df_attempts_normalized_v5.columns
    assert f"{id_col}_key" in df_attempts_normalized_v5.columns

    original = df_attempts_normalized_v5[id_col]
    raw_copy = df_attempts_normalized_v5[f"{id_col}_raw"]

    assert original.equals(raw_copy), f"{id_col}_raw does not preserve original {id_col}"

assert len(df_attempts_normalized_v5) == len(df_attempts_clean_v5)

print("Validation passed: df_attempts_normalized_v5 created and original IDs preserved.")

## 20. Save ID suffix inspection report

This report checks suffix distribution for all ID columns.  
It helps verify suffixes such as `.111` are preserved.


In [ ]:
suffix_reports_v5 = []

for id_col in ID_COLUMNS_V5:
    suffix_col = f"{id_col}_suffix"

    temp = (
        df_attempts_normalized_v5[suffix_col]
        .fillna("<NO_SUFFIX>")
        .value_counts(dropna=False)
        .reset_index()
    )

    temp.columns = ["suffix", "count"]
    temp.insert(0, "id_column", id_col)
    temp["row_count"] = len(df_attempts_normalized_v5)
    temp["ratio"] = temp["count"] / len(df_attempts_normalized_v5)

    suffix_reports_v5.append(temp)

id_suffix_report_v5 = pd.concat(suffix_reports_v5, ignore_index=True)

id_suffix_report_path_v5 = REPORTS_DIR / "id_suffix_report_v5.csv"
id_suffix_report_v5.to_csv(id_suffix_report_path_v5, index=False, encoding="utf-8-sig")

display(id_suffix_report_v5)
print("Saved:", id_suffix_report_path_v5)

In [ ]:
assert id_suffix_report_path_v5.exists()
assert set(id_suffix_report_v5["id_column"]) == set(ID_COLUMNS_V5)

print("Validation passed: ID suffix report saved.")

## 21. Validate normalized academic flags

The normalized dataframe must contain the requested academic flags.


In [ ]:
required_flag_columns_v5 = [
    "is_official_pass",
    "is_transfer_pass",
    "is_internal_performance_attempt",
    "is_fail",
    "is_fail_like",
    "is_withdrawn",
    "is_in_progress",
    "is_archive_or_drop",
    "is_critical_issue",
]

flag_summary_v5 = pd.DataFrame({
    "flag": required_flag_columns_v5,
    "true_count": [int(df_attempts_normalized_v5[col].sum()) for col in required_flag_columns_v5],
    "false_count": [int((~df_attempts_normalized_v5[col]).sum()) for col in required_flag_columns_v5],
    "row_count": len(df_attempts_normalized_v5),
})

flag_summary_v5["true_ratio"] = flag_summary_v5["true_count"] / len(df_attempts_normalized_v5)

display(flag_summary_v5)

In [ ]:
for col in required_flag_columns_v5:
    assert col in df_attempts_normalized_v5.columns, f"Missing flag column: {col}"

assert not df_attempts_normalized_v5["is_archive_or_drop"].any()
assert not df_attempts_normalized_v5["is_critical_issue"].any()
assert not df_attempts_normalized_v5["is_transfer_pass"].any()
assert "is_last_try" not in df_attempts_normalized_v5.columns

print("Validation passed: required flags exist and clean data has no archive/critical rows.")

## 22. Validate that no pass/fail logic comes from marks

This check confirms that official pass/fail flags are based only on `finish_status_clean`.


In [ ]:
expected_is_official_pass_v5 = (
    df_attempts_normalized_v5["finish_status_clean"].isin(PASS_STATUSES_V5)
)

expected_is_fail_v5 = (
    df_attempts_normalized_v5["finish_status_clean"].isin(FAIL_STATUSES_V5)
)

mismatch_official_pass_v5 = (
    df_attempts_normalized_v5["is_official_pass"] != expected_is_official_pass_v5
).sum()

mismatch_fail_v5 = (
    df_attempts_normalized_v5["is_fail"] != expected_is_fail_v5
).sum()

print("Official pass mismatches:", mismatch_official_pass_v5)
print("Fail mismatches:", mismatch_fail_v5)

In [ ]:
assert mismatch_official_pass_v5 == 0, "is_official_pass is not aligned with finish_status rules."
assert mismatch_fail_v5 == 0, "is_fail is not aligned with finish_status rules."

print("Validation passed: pass/fail logic comes only from finish_status.")

## 23. Calculate attempts manually

Because `is_last_try` is not available and was not reliable, attempts are calculated manually.

Logic:
- Group by `student_id_key` + `course_id_key`.
- Sort attempts by `student_course_id_base`.
- `attempt_number_v5` starts from 1.
- `attempt_count_v5` is total attempts per student-course.
- First and last attempt flags are calculated manually.


In [ ]:
student_course_attempt_key_cols_v5 = [
    "student_id_key",
    "course_id_key",
]

df_attempts_normalized_v5 = df_attempts_normalized_v5.sort_values(
    student_course_attempt_key_cols_v5
    + [
        "student_course_id_base",
        "_source_row_number_v5",
    ],
    ascending=[True, True, True, True],
    na_position="last",
).copy()

df_attempts_normalized_v5["attempt_number_v5"] = (
    df_attempts_normalized_v5
    .groupby(student_course_attempt_key_cols_v5, dropna=False)
    .cumcount()
    + 1
)

df_attempts_normalized_v5["attempt_count_v5"] = (
    df_attempts_normalized_v5
    .groupby(student_course_attempt_key_cols_v5, dropna=False)["student_course_id_key"]
    .transform("count")
)

df_attempts_normalized_v5["is_first_attempt_calculated_v5"] = (
    df_attempts_normalized_v5["attempt_number_v5"].eq(1)
)

df_attempts_normalized_v5["is_last_attempt_calculated_v5"] = (
    df_attempts_normalized_v5["attempt_number_v5"]
    .eq(df_attempts_normalized_v5["attempt_count_v5"])
)

display(df_attempts_normalized_v5[[
    "student_id_key",
    "course_id_key",
    "student_course_id",
    "student_course_id_base",
    "finish_status_clean",
    "attempt_number_v5",
    "attempt_count_v5",
    "is_first_attempt_calculated_v5",
    "is_last_attempt_calculated_v5",
]].head(50))

In [ ]:
last_attempt_check_v5 = (
    df_attempts_normalized_v5
    .groupby(student_course_attempt_key_cols_v5, dropna=False)["is_last_attempt_calculated_v5"]
    .sum()
    .reset_index(name="last_attempt_count")
)

first_attempt_check_v5 = (
    df_attempts_normalized_v5
    .groupby(student_course_attempt_key_cols_v5, dropna=False)["is_first_attempt_calculated_v5"]
    .sum()
    .reset_index(name="first_attempt_count")
)

assert last_attempt_check_v5["last_attempt_count"].eq(1).all(), \
    "Each student-course must have exactly one calculated last attempt."

assert first_attempt_check_v5["first_attempt_count"].eq(1).all(), \
    "Each student-course must have exactly one calculated first attempt."

manual_attempts_report_v5 = (
    df_attempts_normalized_v5["attempt_count_v5"]
    .value_counts()
    .sort_index()
    .reset_index()
)

manual_attempts_report_v5.columns = ["attempt_count_v5", "student_course_rows"]
manual_attempts_report_path_v5 = REPORTS_DIR / "manual_attempts_report_v5.csv"
manual_attempts_report_v5.to_csv(
    manual_attempts_report_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(manual_attempts_report_v5)
print("Saved:", manual_attempts_report_path_v5)

assert manual_attempts_report_path_v5.exists()

print("Validation passed: manual attempt calculation is correct.")

## 24. Create df_student_course_current_status_v5

Current status is created from the calculated last attempt only.  
No `is_last_try` column is used.


In [ ]:
df_student_course_current_status_v5 = (
    df_attempts_normalized_v5[
        df_attempts_normalized_v5["is_last_attempt_calculated_v5"]
    ]
    .copy()
)

df_student_course_current_status_v5["current_status_v5"] = np.select(
    [
        df_student_course_current_status_v5["finish_status_clean"].isin(PASS_STATUSES_V5),
        df_student_course_current_status_v5["finish_status_clean"].isin(FAIL_STATUSES_V5),
        df_student_course_current_status_v5["finish_status_clean"].isin(WITHDRAWN_STATUSES_V5),
    ],
    [
        "passed",
        "failed",
        "withdrawn",
    ],
    default="review",
)

df_student_course_current_status_v5["current_is_passed_v5"] = (
    df_student_course_current_status_v5["finish_status_clean"].isin(PASS_STATUSES_V5)
)

df_student_course_current_status_v5["current_is_failed_v5"] = (
    df_student_course_current_status_v5["finish_status_clean"].isin(FAIL_STATUSES_V5)
)

df_student_course_current_status_v5["current_is_withdrawn_v5"] = (
    df_student_course_current_status_v5["finish_status_clean"].isin(WITHDRAWN_STATUSES_V5)
)

df_student_course_current_status_v5["current_is_transfer_v5"] = False
df_student_course_current_status_v5["current_is_in_progress_v5"] = False

display(df_student_course_current_status_v5.head())

In [ ]:
current_duplicates_v5 = df_student_course_current_status_v5.duplicated(
    subset=student_course_attempt_key_cols_v5
).sum()

print("Current status rows:", len(df_student_course_current_status_v5))
print("Duplicate student-course current rows:", current_duplicates_v5)

assert current_duplicates_v5 == 0, "Current status has duplicate student-course rows."
assert df_student_course_current_status_v5["student_id_key"].notna().all()
assert df_student_course_current_status_v5["course_id_key"].notna().all()
assert "is_last_try" not in df_student_course_current_status_v5.columns

print("Validation passed: one current status per student-course combination.")

## 25. Current status distribution

This visualization checks the final current academic status distribution.


In [ ]:
current_status_distribution_v5 = (
    df_student_course_current_status_v5["current_status_v5"]
    .value_counts(dropna=False)
    .reset_index()
)

current_status_distribution_v5.columns = ["current_status_v5", "count"]
current_status_distribution_v5["row_count"] = len(df_student_course_current_status_v5)
current_status_distribution_v5["ratio"] = (
    current_status_distribution_v5["count"] / len(df_student_course_current_status_v5)
)

current_status_distribution_path_v5 = REPORTS_DIR / "current_status_distribution_v5.csv"
current_status_distribution_v5.to_csv(
    current_status_distribution_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(current_status_distribution_v5)
print("Saved:", current_status_distribution_path_v5)

In [ ]:
assert current_status_distribution_path_v5.exists()
assert current_status_distribution_v5["count"].sum() == len(df_student_course_current_status_v5)

print("Validation passed: current status distribution is consistent.")

In [ ]:
plot_df = current_status_distribution_v5.sort_values("count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["current_status_v5"], plot_df["count"])
plt.title("Current Student-Course Status Distribution — V5")
plt.xlabel("Rows")
plt.ylabel("Current Status")
plt.tight_layout()
plt.show()

## 26. Save final outputs

Main storage format is Parquet.  
CSV is saved as optional inspection export.


In [ ]:
df_attempts_clean_parquet_path_v5 = (
    PREPROCESSED_DIR / "df_attempts_clean_v5.parquet"
)

df_attempts_clean_csv_path_v5 = (
    PREPROCESSED_DIR / "df_attempts_clean_v5.csv"
)

df_attempts_normalized_parquet_path_v5 = (
    PREPROCESSED_DIR / "df_attempts_normalized_v5.parquet"
)

df_attempts_normalized_csv_path_v5 = (
    PREPROCESSED_DIR / "df_attempts_normalized_v5.csv"
)

df_current_status_parquet_path_v5 = (
    FEATURES_DIR / "df_student_course_current_status_v5.parquet"
)

df_current_status_csv_path_v5 = (
    FEATURES_DIR / "df_student_course_current_status_v5.csv"
)

df_attempts_clean_v5.to_parquet(df_attempts_clean_parquet_path_v5, index=False)
df_attempts_clean_v5.to_csv(df_attempts_clean_csv_path_v5, index=False, encoding="utf-8-sig")

df_attempts_normalized_v5.to_parquet(df_attempts_normalized_parquet_path_v5, index=False)
df_attempts_normalized_v5.to_csv(df_attempts_normalized_csv_path_v5, index=False, encoding="utf-8-sig")

df_student_course_current_status_v5.to_parquet(df_current_status_parquet_path_v5, index=False)
df_student_course_current_status_v5.to_csv(df_current_status_csv_path_v5, index=False, encoding="utf-8-sig")

print("Saved:", df_attempts_clean_parquet_path_v5)
print("Saved:", df_attempts_clean_csv_path_v5)
print("Saved:", df_attempts_normalized_parquet_path_v5)
print("Saved:", df_attempts_normalized_csv_path_v5)
print("Saved:", df_current_status_parquet_path_v5)
print("Saved:", df_current_status_csv_path_v5)

In [ ]:
required_output_paths_v5 = [
    df_attempts_clean_parquet_path_v5,
    df_attempts_clean_csv_path_v5,
    df_attempts_normalized_parquet_path_v5,
    df_attempts_normalized_csv_path_v5,
    df_current_status_parquet_path_v5,
    df_current_status_csv_path_v5,
]

for path in required_output_paths_v5:
    assert path.exists(), f"Missing output file: {path}"

print("Validation passed: all final output files saved.")

## 27. Save final validation summary report

This report summarizes the main validation checks and final row counts.


In [ ]:
validation_summary_v5 = pd.DataFrame([
    {
        "check_name": "raw_file_exists",
        "status": RAW_PATH.exists(),
        "value": str(RAW_PATH),
    },
    {
        "check_name": "raw_rows",
        "status": len(df_raw_v5) > 0,
        "value": len(df_raw_v5),
    },
    {
        "check_name": "is_last_try_not_used",
        "status": "is_last_try" not in df_raw_v5.columns,
        "value": "is_last_try removed from required columns and workflow",
    },
    {
        "check_name": "required_columns_exist",
        "status": len(missing_required_columns_v5) == 0,
        "value": str(missing_required_columns_v5),
    },
    {
        "check_name": "final_mark_integer_like",
        "status": final_mark_fractional == 0,
        "value": final_mark_fractional,
    },
    {
        "check_name": "course_credits_fractional_rows_reported",
        "status": fractional_course_credits_rows_path_v5.exists(),
        "value": int(course_credits_fractional),
    },
    {
        "check_name": "exact_duplicates_reported",
        "status": duplicate_rows_path_v5.exists(),
        "value": len(df_duplicate_rows_v5),
    },
    {
        "check_name": "null_handling_report_saved",
        "status": null_handling_report_path_v5.exists(),
        "value": str(null_handling_report_path_v5),
    },
    {
        "check_name": "finish_status_null_rows_archived",
        "status": not df_attempts_clean_v5["finish_status_clean"].isna().any(),
        "value": int(df_flagged_v5["finish_status_is_null_v5"].sum()),
    },
    {
        "check_name": "course_credits_null_rows_reported_not_filled",
        "status": course_credits_null_report_path_v5.exists(),
        "value": len(course_credits_null_rows_v5),
    },
    {
        "check_name": "archive_drop_rows_flagged",
        "status": archive_drop_report_path_v5.exists(),
        "value": int(df_flagged_v5["is_archive_or_drop"].sum()),
    },
    {
        "check_name": "critical_issue_rows_flagged",
        "status": critical_issues_report_path_v5.exists(),
        "value": int(df_flagged_v5["is_critical_issue"].sum()),
    },
    {
        "check_name": "clean_rows",
        "status": len(df_attempts_clean_v5) > 0,
        "value": len(df_attempts_clean_v5),
    },
    {
        "check_name": "clean_statuses_are_only_p_f_fa_fe_w",
        "status": df_attempts_clean_v5["finish_status_clean"].isin(KEEP_STATUSES_V5).all(),
        "value": sorted(df_attempts_clean_v5["finish_status_clean"].dropna().unique().tolist()),
    },
    {
        "check_name": "normalized_rows_equal_clean_rows",
        "status": len(df_attempts_normalized_v5) == len(df_attempts_clean_v5),
        "value": f"{len(df_attempts_normalized_v5)} == {len(df_attempts_clean_v5)}",
    },
    {
        "check_name": "manual_attempts_calculated",
        "status": (
            "attempt_number_v5" in df_attempts_normalized_v5.columns
            and "is_last_attempt_calculated_v5" in df_attempts_normalized_v5.columns
        ),
        "value": "attempt_number_v5 / attempt_count_v5 / calculated first-last flags",
    },
    {
        "check_name": "current_status_one_row_per_student_course",
        "status": current_duplicates_v5 == 0,
        "value": current_duplicates_v5,
    },
    {
        "check_name": "no_pass_logic_from_final_mark",
        "status": mismatch_official_pass_v5 == 0,
        "value": mismatch_official_pass_v5,
    },
    {
        "check_name": "df_attempts_clean_v5_saved",
        "status": df_attempts_clean_parquet_path_v5.exists(),
        "value": str(df_attempts_clean_parquet_path_v5),
    },
    {
        "check_name": "df_attempts_normalized_v5_saved",
        "status": df_attempts_normalized_parquet_path_v5.exists(),
        "value": str(df_attempts_normalized_parquet_path_v5),
    },
    {
        "check_name": "df_student_course_current_status_v5_saved",
        "status": df_current_status_parquet_path_v5.exists(),
        "value": str(df_current_status_parquet_path_v5),
    },
])

validation_summary_path_v5 = REPORTS_DIR / "validation_summary_v5.csv"
validation_summary_v5.to_csv(validation_summary_path_v5, index=False, encoding="utf-8-sig")

display(validation_summary_v5)
print("Saved:", validation_summary_path_v5)

In [ ]:
assert validation_summary_path_v5.exists()
assert validation_summary_v5["status"].all(), "Some validation checks failed."

print("V5 preprocessing completed successfully.")
print("Created: df_attempts_clean_v5")
print("Created: df_attempts_normalized_v5")
print("Created: df_student_course_current_status_v5")
print("No is_last_try used.")
print("Manual attempt calculation completed.")
print("No models built.")
print("No course difficulty built.")
print("No recommendation logic built.")